# Stage 1 MVP: Road Freight Delivery Sequence Optimization

This Google Colab notebook runs the Stage 1 MVP only:

- Single-route, single-vehicle delivery stop sequencing.
- Amazon Last Mile Routing Research Challenge Dataset.
- One baseline: Nearest Neighbour.
- One ML model: Random Forest next-stop prediction.

The notebook expects the dataset files `route_data.json`, `actual_sequences.json`, `package_data.json`, and `travel_times.json`. The full `travel_times.json` is not loaded into memory; only selected route IDs are streamed with `ijson`.

In [ ]:
# Colab setup: works both from a checked-out repo and when opened directly from GitHub.
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Chen0421-c/delivery-sequence-ml.git"
REPO_DIR = Path("/content/delivery-sequence-ml")

if not Path("requirements.txt").exists():
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)

print("Notebook working directory:", Path.cwd())
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    # Update this path if you cloned the repository elsewhere in Colab.
    REPO_ROOT = Path("/content/delivery-sequence-ml")

sys.path.insert(0, str(REPO_ROOT))
print("Repository root:", REPO_ROOT)

## Locate or provide the dataset

Place the Amazon dataset under `data/` or set `DATA_DIR` to a Google Drive folder or uploaded folder that contains the JSON files. The repository `.gitignore` excludes `data/` because the dataset is large.

In [ ]:
# Optional Google Drive mount.
# from google.colab import drive
# drive.mount('/content/drive')

DATA_DIR = REPO_ROOT / "data"  # Change this to your dataset folder when needed.
N_ROUTES = 100                 # Stage 1 default small working subset.
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"

required = ["route_data.json", "actual_sequences.json", "package_data.json", "travel_times.json"]
print("Dataset folder:", DATA_DIR)
print("Expected files:", required)

## Memory-safe load and preprocessing

This step streams the top-level route IDs and loads only the selected subset from every dataset file, including `travel_times.json`.

In [ ]:
from src.data_loader import load_stage1_subset
from src.preprocessing import filter_valid_routes, build_stop_feature_table

dataset = load_stage1_subset(DATA_DIR, n_routes=N_ROUTES)
route_data = dataset["route_data"]
actual_sequences = dataset["actual_sequences"]
package_data = dataset["package_data"]
travel_times = dataset["travel_times"]

valid_route_ids, invalid_report = filter_valid_routes(route_data, actual_sequences, package_data, travel_times)
print(f"Loaded routes: {len(route_data)}")
print(f"Valid routes: {len(valid_route_ids)}")
print(f"Invalid routes: {len(invalid_report)}")
display(invalid_report.head())

stop_features = build_stop_feature_table(valid_route_ids, route_data, actual_sequences, package_data)
display(stop_features.head())

## Train Random Forest with a route-level split

Candidate rows are generated from route sequences, but train/test split is performed by `route_id` to avoid route leakage.

In [ ]:
from src.ml_model import (
    build_next_stop_training_samples,
    next_stop_accuracy,
    route_level_train_test_split,
    train_random_forest_model,
)

train_route_ids, test_route_ids = route_level_train_test_split(valid_route_ids, test_size=0.2, random_state=42)
if not test_route_ids:
    test_route_ids = train_route_ids

train_samples = build_next_stop_training_samples(train_route_ids, route_data, actual_sequences, package_data, travel_times)
test_samples = build_next_stop_training_samples(test_route_ids, route_data, actual_sequences, package_data, travel_times)

model = train_random_forest_model(train_samples, random_state=42)
accuracy = next_stop_accuracy(model, test_samples)
print(f"Train routes: {len(train_route_ids)}")
print(f"Test routes: {len(test_route_ids)}")
print(f"Top-1 next-stop prediction accuracy: {accuracy:.4f}")
display(train_samples.head())

## Evaluate actual, Nearest Neighbour, and Random Forest sequences

In [ ]:
import pandas as pd
from src.baseline import nearest_neighbour_sequence
from src.evaluation import evaluate_sequence
from src.ml_model import generate_rf_sequence
from src.preprocessing import get_actual_sequence, get_depot_stop_id, get_route_stops

rows = []
example_sequences = {}
for route_id in test_route_ids:
    actual_sequence = get_actual_sequence(route_id, actual_sequences)
    depot_id = get_depot_stop_id(route_id, route_data, actual_sequences)
    stop_ids = list(get_route_stops(route_id, route_data))
    matrix = travel_times[route_id]

    nn_sequence = nearest_neighbour_sequence(stop_ids, depot_id, matrix)
    rf_sequence = generate_rf_sequence(route_id, model, route_data, actual_sequences, package_data, travel_times)

    rows.append(evaluate_sequence(route_id, "actual_historical", actual_sequence, actual_sequence, matrix))
    rows.append(evaluate_sequence(route_id, "nearest_neighbour", actual_sequence, nn_sequence, matrix))
    rows.append(evaluate_sequence(route_id, "random_forest", actual_sequence, rf_sequence, matrix))
    example_sequences = {"route_id": route_id, "actual": actual_sequence, "nearest_neighbour": nn_sequence, "random_forest": rf_sequence}

evaluation = pd.DataFrame(rows)
evaluation["next_stop_prediction_accuracy"] = accuracy
summary = evaluation.groupby("method", as_index=False).mean(numeric_only=True)

RESULTS_DIR.mkdir(exist_ok=True, parents=True)
evaluation.to_csv(RESULTS_DIR / "evaluation.csv", index=False)
summary.to_csv(RESULTS_DIR / "evaluation_summary.csv", index=False)

display(evaluation.head())
display(summary)

## Visualize one route with Folium

In [ ]:
from IPython.display import display
from src.visualization import plot_route_folium

FIGURES_DIR.mkdir(exist_ok=True, parents=True)
route_id = example_sequences["route_id"]
actual_map = plot_route_folium(route_id, example_sequences["actual"], route_data, FIGURES_DIR / f"{route_id}_actual.html")
plot_route_folium(route_id, example_sequences["nearest_neighbour"], route_data, FIGURES_DIR / f"{route_id}_nearest_neighbour.html")
plot_route_folium(route_id, example_sequences["random_forest"], route_data, FIGURES_DIR / f"{route_id}_random_forest.html")

print("Saved maps to", FIGURES_DIR)
display(actual_map)

## Optional: run the packaged pipeline

The modular pipeline writes preprocessing outputs, evaluation CSVs, and maps to `results/` and `figures/`.

In [ ]:
# from src.stage1_pipeline import run_stage1
# outputs = run_stage1(DATA_DIR, n_routes=N_ROUTES, results_dir=RESULTS_DIR, figures_dir=FIGURES_DIR)
# display(outputs["summary"])